# Phase 4 — Mixture Models: Exploration Notebook

This notebook explores Gaussian Mixture Models:
- Fit GMMs to bimodal salary data
- Visualize EM convergence
- BIC-based K selection
- Multimodality detection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data_gen import generate_mixed_data
from src.distributions import NormalDist
from src.mixture import fit_gmm, select_K, detect_multimodality

sns.set_theme(style="whitegrid", font_scale=1.1)
SEED = 42

## 1. Generate Bimodal Salary Data

In [ ]:
junior = NormalDist(mu=8000, sigma=1500)
senior = NormalDist(mu=18000, sigma=2500)
data = generate_mixed_data(1000, [junior, senior], [0.6, 0.4], seed=SEED)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(data, bins=50, density=True, alpha=0.6, color="steelblue")
ax.set_xlabel("Salary (R$)")
ax.set_ylabel("Density")
ax.set_title("Bimodal Salary Data: 60% Junior + 40% Senior")
plt.tight_layout()
plt.show()

print(f"n={len(data)}, mean={data.mean():.0f}, std={data.std():.0f}")

## 2. Fit GMM with K=2

In [ ]:
result = fit_gmm(data, K=2, seed=SEED)

order = np.argsort(result.means)
print("GMM Fit (K=2)")
print("=" * 50)
for idx, k in enumerate(order):
    print(f"Component {idx+1}:")
    print(f"  weight = {result.weights[k]:.3f} (true: {[0.6, 0.4][idx]})")
    print(f"  mean   = {result.means[k]:.0f} (true: {[8000, 18000][idx]})")
    print(f"  std    = {np.sqrt(result.variances[k]):.0f} (true: {[1500, 2500][idx]})")
print(f"\nLog-likelihood: {result.loglik:.1f}")
print(f"BIC: {result.bic:.1f}")
print(f"Converged: {result.converged} (in {result.n_iter} iterations)")

## 3. Visualize Components

In [ ]:
x = np.linspace(data.min() - 1000, data.max() + 1000, 500)

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(data, bins=50, density=True, alpha=0.3, color="gray", label="Data")

colors = ["#2A9D8F", "#E63946"]
total = np.zeros_like(x)
for k in range(result.K):
    comp = result.weights[k] * stats.norm.pdf(x, result.means[k], np.sqrt(result.variances[k]))
    total += comp
    ax.plot(x, comp, linewidth=2, color=colors[k],
            label=f"Component {k+1} (w={result.weights[k]:.2f})")

ax.plot(x, total, "k-", linewidth=2.5, label="Mixture PDF")
ax.set_xlabel("Salary (R$)")
ax.set_ylabel("Density")
ax.set_title("GMM Fit: 2-Component Gaussian Mixture")
ax.legend()
plt.tight_layout()
plt.show()

## 4. BIC-Based K Selection

In [ ]:
optimal_K, results = select_K(data, K_range=[1, 2, 3, 4, 5], seed=SEED)

bics = [r.bic for r in results]
Ks = [1, 2, 3, 4, 5]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(Ks, bics, "o-", markersize=10, linewidth=2, color="#457B9D")
ax.axvline(optimal_K, color="red", linestyle="--", label=f"Optimal K={optimal_K}")
ax.set_xlabel("Number of Components (K)")
ax.set_ylabel("BIC")
ax.set_title("BIC vs Number of Components")
ax.set_xticks(Ks)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Optimal K: {optimal_K}")
for K, bic in zip(Ks, bics):
    marker = " <-- best" if K == optimal_K else ""
    print(f"  K={K}: BIC={bic:.1f}{marker}")

## 5. Multimodality Detection

In [ ]:
# Bimodal data
result_bimodal = detect_multimodality(data, seed=SEED)
print("Bimodal data:")
print(f"  Multimodal: {result_bimodal.is_multimodal}")
print(f"  {result_bimodal.evidence}")

# Unimodal data for comparison
rng = np.random.default_rng(SEED)
unimodal = rng.lognormal(9.1, 0.4, size=1000)
result_unimodal = detect_multimodality(unimodal, seed=SEED)
print(f"\nUnimodal data:")
print(f"  Multimodal: {result_unimodal.is_multimodal}")
print(f"  {result_unimodal.evidence}")

## 6. Budget Impact: Single Normal vs GMM

In [ ]:
# Single Normal fit
mu_single = data.mean()
sigma_single = data.std()

# Budget at 95th percentile
budget_normal = mu_single + 1.645 * sigma_single

# GMM-based budget: simulate from the mixture and take 95th percentile
rng = np.random.default_rng(SEED)
gmm_samples = []
for _ in range(100_000):
    k = rng.choice(result.K, p=result.weights)
    s = rng.normal(result.means[k], np.sqrt(result.variances[k]))
    gmm_samples.append(s)
gmm_samples = np.array(gmm_samples)
budget_gmm = np.percentile(gmm_samples, 95)

print("Budget at 95th Percentile")
print("=" * 40)
print(f"Single Normal: R${budget_normal:,.0f}")
print(f"GMM (K={result.K}): R${budget_gmm:,.0f}")
print(f"Difference: R${budget_gmm - budget_normal:+,.0f}")